# Day 16 — pytest Basics

> ⚠️ **Why this matters.** Until now you've tested by running the code and looking at output. That doesn't scale. Tests let you say with confidence: 'this still works' — even after a 200-line refactor. Today you add real tests to english-helper.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jakkzz/prince-curriculum/blob/main/phase-1-python-cli/lessons/16-pytest-basics.ipynb)

## What you'll do today

- [ ] You can install + run pytest
- [ ] You can write tests using `assert` and `pytest.raises`
- [ ] You can parametrize tests
- [ ] You've added ~10 tests for `Word` and `WordStore`
- [ ] `uv run pytest` runs green on your project

## 1. Install + first test

In [ ]:
# In your english-helper project:
# $ uv add --dev pytest
#
# Create tests/test_word.py:
from english_helper.word import Word

def test_word_creation():
    w = Word('thorough', '/ˈθʌrə/', 'ละเอียด')
    assert w.word == 'thorough'
    assert w.ipa == '/ˈθʌrə/'
    assert w.thai == 'ละเอียด'

def test_word_default_fields():
    w = Word('cat')
    assert w.ipa == ''
    assert w.thai == ''

Run it:

```bash
$ uv run pytest
================== test session starts ==================
collected 2 items

tests/test_word.py ..                              [100%]

================== 2 passed in 0.04s ====================
```

**Rules pytest follows:**

- Test files must start with `test_` or end with `_test.py`
- Test functions must start with `test_`
- `assert` raises AssertionError on failure; pytest catches and reports it
- The output `...` shows test status: `.` pass, `F` fail, `E` error, `s` skipped

## 2. Asserting exceptions

In [ ]:
import pytest
from english_helper.dictionary import strict_lookup, WordNotFoundError

def test_strict_lookup_raises_when_missing():
    with pytest.raises(WordNotFoundError):
        strict_lookup([], 'nope')

def test_strict_lookup_raises_with_message():
    with pytest.raises(WordNotFoundError, match='nope'):
        strict_lookup([], 'nope')

`pytest.raises(ExceptionClass)` — test passes only if the block raises that exception. `match=` checks the exception's message against a regex.

## 3. Parametrize — same test, many inputs

In [ ]:
import pytest

@pytest.mark.parametrize('text,expected', [
    ('thorough', 8),
    ('cat', 3),
    ('', 0),
])
def test_word_length(text, expected):
    assert len(text) == expected

pytest runs this test once per row, with friendly names:

```
test_word_length[thorough-8] PASSED
test_word_length[cat-3] PASSED
test_word_length[-0] PASSED
```

Far better than copy-pasting the same test body 3 times.

## 4. Test organization

Match your `src/` layout in `tests/`:

```
english-helper/
├── src/english_helper/
│   ├── word.py
│   ├── storage.py
│   └── quiz.py
└── tests/
    ├── test_word.py        # tests src/english_helper/word.py
    ├── test_storage.py     # tests src/english_helper/storage.py
    └── test_quiz.py        # tests src/english_helper/quiz.py
```

**One test file per source module** is the simple rule. Some projects split further (one file per class) — only do that when test files exceed ~300 lines.

## 5. Useful pytest flags

| Flag | What |
|------|------|
| `pytest` | Run all tests |
| `pytest -v` | Verbose (show each test name) |
| `pytest -x` | Stop on first failure |
| `pytest -k pattern` | Run only tests matching pattern |
| `pytest tests/test_word.py` | Run only this file |
| `pytest tests/test_word.py::test_word_creation` | Run one test |
| `pytest --pdb` | Drop into debugger on failure |
| `pytest -s` | Don't capture stdout (you'll see your prints) |


## End-of-day mini-project — `tests/test_word.py` + `tests/test_storage.py`

> 🎯 **Today's piece:** Add ~10 tests covering Word and WordStore.

### Required tests

**`tests/test_word.py`** (5+ tests):

- `test_creation_with_all_fields`
- `test_creation_with_defaults`
- `test_equality` — two Words with same fields are equal
- `test_inequality` — different fields → unequal
- `test_hashable` — can put in a set
- `test_to_dict_roundtrip` — `Word.from_dict(w.to_dict()) == w`

**`tests/test_storage.py`** (5+ tests):

- `test_empty_store`
- `test_add_and_lookup`
- `test_add_duplicate_overrides` (or raises, depending on your design)
- `test_remove_existing` — returns True, word gone
- `test_remove_nonexistent` — returns False
- `test_save_and_load_roundtrip` — use a tmp file

### Verify

```bash
$ uv run pytest -v
tests/test_word.py::test_creation_with_all_fields PASSED
tests/test_word.py::test_creation_with_defaults PASSED
...
============ 11 passed in 0.12s ===========
```

**Note about `test_save_and_load_roundtrip`:** Use `tmp_path` fixture (built into pytest):
```python
def test_save_and_load(tmp_path):
    path = tmp_path / 'test.json'
    store = WordStore(path=path)
    store.add(Word('thorough', '/x/'))
    store.save()
    loaded = WordStore.load(path=path)
    assert loaded.lookup('thorough') == Word('thorough', '/x/')
```

## Connect to the project

> 🎯 **Tomorrow (Day 17):** Test-Driven Development — write the test first, then the code. Feels weird; works very well.

**Quiz:** [16-pytest-basics-quiz.ipynb](16-pytest-basics-quiz.ipynb)